In [37]:
# ================== Install Dependencies (for Colab or first-time setup) ==================
# Run this cell only once if libraries are missing
!pip install requests beautifulsoup4 pandas openpyxl lxml regex


In [38]:
# Norway Agricultural Subsidy Zone Mapping for Google Colab
# This script scrapes subsidy zone data and adds it to the main dataset

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from difflib import SequenceMatcher

print("=" * 80)
print("Norway Agricultural Subsidy Zone Mapping System")
print("=" * 80)


Norway Agricultural Subsidy Zone Mapping System


In [39]:
# ================== Part 1: Web Scraping ==================

def scrape_subsidy_zones():
    """Scrape Norwegian agricultural subsidy zone data from website"""
    print("\n[Step 1] Scraping subsidy zone data...")

    url = "https://www.landbruksdirektoratet.no/nb/jordbruk/kart-og-register/soner-for-arealtilskudd"

    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    data = []
    tables = soup.find_all("table")

    for table in tables:
        county_tag = table.find_previous(["h2", "strong"])
        county = county_tag.text.strip() if county_tag else "Unknown"

        rows = table.find_all("tr")[1:]  # Skip header
        for row in rows:
            cols = [c.text.strip() for c in row.find_all("td")]
            if len(cols) == 3:
                data.append({
                    "County": county,
                    "Kommunenummer": cols[0],
                    "Kommune": cols[1],
                    "Sone": cols[2]
                })

    df = pd.DataFrame(data)
    print(f"✓ Successfully scraped {len(df)} rows of subsidy zone data")
    return df


In [40]:
# ================== Part 2: Kommune Matching Functions ==================

def clean_kommune_name(name):
    """Clean Kommune name for better matching"""
    if pd.isna(name):
        return ""
    # Convert to string and remove spaces
    name = str(name).strip()
    # Convert to lowercase for comparison
    return name.lower()

def normalize_kommune_name(name):
    """Normalize Kommune name to improve matching"""
    if pd.isna(name):
        return ""
    name = str(name).strip()
    # Handling special characters
    name = name.replace('-', ' ')
    name = name.replace('/', ' ')
    # Standardizing spaces
    name = re.sub(r'\s+', ' ', name)
    return name.lower()

def extract_main_name(name):
    """Extract the primary name (excluding content within brackets)"""
    if pd.isna(name):
        return ""
    name = str(name).strip()
    # Remove parentheses and their content
    main_name = re.sub(r'\s*\([^)]*\)', '', name)
    return main_name.strip()

def calculate_similarity(str1, str2):
    """Calculate the similarity between two strings"""
    return SequenceMatcher(None, str1.lower(), str2.lower()).ratio()


In [41]:
# ================== Part 3: Kommune Mapping Analysis ==================

def match_kommune_data(main_data, subsidy_zones):
    """
    Execute Kommune matching analysis
    """
    print("\n[Step 2] Starting Kommune matching analysis...")

    print(f"Main_Data_Bank contains {len(main_data)} rows of data")
    print(f"norway_subsidy_zones contains {len(subsidy_zones)} rows of data")

    # Get unique Kommune lists
    main_kommuner = main_data['Kommune'].dropna().unique()
    subsidy_kommuner = subsidy_zones['Kommune'].unique()

    print(f"\nUnique Kommunes in Main_Data_Bank: {len(main_kommuner)}")
    print(f"Unique Kommunes in norway_subsidy_zones: {len(subsidy_kommuner)}")

    # Create dictionary to hold match results
    match_results = {
        'exact_matches': [],
        'fuzzy_matches': [],
        'no_matches': []
    }

    # Create lookup dictionary for subsidy_zones (with multiple forms)
    subsidy_dict = {}
    subsidy_main_dict = {}
    subsidy_normalized_dict = {}

    for _, row in subsidy_zones.iterrows():
        kommune = row['Kommune']
        kommune_lower = clean_kommune_name(kommune)
        kommune_main = extract_main_name(kommune).lower()
        kommune_normalized = normalize_kommune_name(kommune)

        subsidy_dict[kommune_lower] = row
        if kommune_main:
            subsidy_main_dict[kommune_main] = row
        if kommune_normalized:
            subsidy_normalized_dict[kommune_normalized] = row

    # Execute matching
    print("\nStarting Kommune matching...")
    for kommune in main_kommuner:
        kommune_clean = clean_kommune_name(kommune)
        kommune_main = extract_main_name(kommune).lower()
        kommune_normalized = normalize_kommune_name(kommune)

        if kommune_clean == "":
            continue

        matched = False
        match_info = None

        # 1. Attempt exact match
        if kommune_clean in subsidy_dict:
            match_info = {
                'main_kommune': kommune,
                'matched_kommune': subsidy_dict[kommune_clean]['Kommune'],
                'sone': subsidy_dict[kommune_clean]['Sone'],
                'county': subsidy_dict[kommune_clean]['County'],
                'kommunenummer': subsidy_dict[kommune_clean]['Kommunenummer']
            }
            match_results['exact_matches'].append(match_info)
            matched = True

        # 2. Special handling of certain known matching patterns
        if not matched:
            special_matches = {
                'deatnu tana': 'Deatnu-Tana',
                'unjargga nesseby': 'Unjárga-Nesseby',
                'porsanger porsángu porsanki': 'Porsanger-Porsáŋgu-Porsanki',
                'guovdageaidnu kautokeino': 'Guovdageaidnu-Kautokeino',
                'karasjohka karasjok': 'Kárášjohka-Karasjok',
                'gáivuotna kåfjord': 'Gáivuotna-Kåfjord-Kaivuono',
                'fauske': 'Fauske-Fuossko',
                'harstad': 'Harstad - Hárstták',
                'lavangen': 'Loabák - Lavangen',
                'røyrvik': 'Raarvihke - Røyrvik',
                'snåsa': 'Snåase-Snåsa',
                'storfjord': 'Storfjord-Omasvuotna-Omasvuono',
            }

            if kommune_clean in special_matches:
                target_kommune = special_matches[kommune_clean]
                for _, row in subsidy_zones.iterrows():
                    if row['Kommune'] == target_kommune:
                        match_info = {
                            'main_kommune': kommune,
                            'matched_kommune': row['Kommune'],
                            'sone': row['Sone'],
                            'county': row['County'],
                            'kommunenummer': row['Kommunenummer'],
                            'match_reason': 'Special matching rule',
                            'similarity': calculate_similarity(kommune, row['Kommune'])
                        }
                        match_results['fuzzy_matches'].append(match_info)
                        matched = True
                        break

        # 3. Handling municipalities with 'unntatt tidligere' or 'tidligere'
        if not matched and kommune_clean in ['heim', 'inderøy', 'indre fosen', 'steinkjer']:
            # These Kommune may have multiple entries within subsidy_zones
            # We choose the first matching entry
            for _, row in subsidy_zones.iterrows():
                if row['Kommune'].lower().startswith(kommune_clean):
                    match_info = {
                        'main_kommune': kommune,
                        'matched_kommune': row['Kommune'],
                        'sone': row['Sone'],
                        'county': row['County'],
                        'kommunenummer': row['Kommunenummer'],
                        'match_reason': 'Partial Kommune match (with tidligere)',
                        'similarity': calculate_similarity(kommune, row['Kommune'].split(',')[0])
                    }
                    match_results['fuzzy_matches'].append(match_info)
                    matched = True
                    break

        # 4. Special handling for Våler and Herøy
        if not matched and kommune_main in ['våler', 'herøy']:
            # Match the corresponding county based on the content in parentheses
            if '(innlandet)' in kommune_clean or 'våler (innlandet)' == kommune:
                target = 'Våler (Hedm.)Våler (Hedm.)'  # Innlandet Våler
            elif '(viken)' in kommune_clean or 'våler (viken)' == kommune:
                target = 'Våler (Østf.)'  # Østfold Våler
            elif '(møre og romsdal)' in kommune_clean or 'herøy (møre og romsdal)' == kommune:
                target = 'Herøy (M. og R.)'
            elif '(nordland)' in kommune_clean or 'herøy (nordland)' == kommune:
                target = 'Herøy (Nordl.)'
            else:
                target = None

            if target:
                for _, row in subsidy_zones.iterrows():
                    if row['Kommune'] == target:
                        match_info = {
                            'main_kommune': kommune,
                            'matched_kommune': row['Kommune'],
                            'sone': row['Sone'],
                            'county': row['County'],
                            'kommunenummer': row['Kommunenummer'],
                            'match_reason': 'County specific match',
                            'similarity': calculate_similarity(kommune_main, extract_main_name(row['Kommune']))
                        }
                        match_results['fuzzy_matches'].append(match_info)
                        matched = True
                        break

        # 5. Attempt main name match (without parentheses)
        if not matched and kommune_main and kommune_main in subsidy_main_dict:
            matched_row = subsidy_main_dict[kommune_main]
            match_info = {
                'main_kommune': kommune,
                'matched_kommune': matched_row['Kommune'],
                'sone': matched_row['Sone'],
                'county': matched_row['County'],
                'kommunenummer': matched_row['Kommunenummer'],
                'match_reason': 'Main name match',
                'similarity': calculate_similarity(kommune, matched_row['Kommune'])
            }
            match_results['fuzzy_matches'].append(match_info)
            matched = True

        # 6. Attempt normalized name match
        if not matched and kommune_normalized in subsidy_normalized_dict:
            matched_row = subsidy_normalized_dict[kommune_normalized]
            match_info = {
                'main_kommune': kommune,
                'matched_kommune': matched_row['Kommune'],
                'sone': matched_row['Sone'],
                'county': matched_row['County'],
                'kommunenummer': matched_row['Kommunenummer'],
                'match_reason': 'Normalized name match',
                'similarity': calculate_similarity(kommune, matched_row['Kommune'])
            }
            match_results['fuzzy_matches'].append(match_info)
            matched = True

        # 7. Attempt high similarity fuzzy match
        if not matched:
            best_match = None
            best_similarity = 0

            for _, row in subsidy_zones.iterrows():
                subsidy_kommune = row['Kommune']

                # Calculate multiple similarity scores
                similarities = [
                    calculate_similarity(kommune, subsidy_kommune),
                    calculate_similarity(kommune_main, extract_main_name(subsidy_kommune)),
                    calculate_similarity(kommune_normalized, normalize_kommune_name(subsidy_kommune))
                ]

                max_similarity = max(similarities)

                if max_similarity > best_similarity and max_similarity > 0.75:
                    best_similarity = max_similarity
                    best_match = row

            if best_match is not None:
                match_info = {
                    'main_kommune': kommune,
                    'matched_kommune': best_match['Kommune'],
                    'sone': best_match['Sone'],
                    'county': best_match['County'],
                    'kommunenummer': best_match['Kommunenummer'],
                    'match_reason': 'High similarity match',
                    'similarity': best_similarity
                }
                match_results['fuzzy_matches'].append(match_info)
                matched = True

        # If still no match
        if not matched:
            match_results['no_matches'].append(kommune)

    # Output match result statistics
    print("\nMatch results summary:")
    print(f"Exact matches: {len(match_results['exact_matches'])}")
    print(f"Fuzzy matches: {len(match_results['fuzzy_matches'])}")
    print(f"No matches: {len(match_results['no_matches'])}")

    # Create detailed DataFrame of matching results
    all_matches = []

    # Add exact matches
    for match in match_results['exact_matches']:
        match['match_type'] = 'Exact match'
        match['confidence'] = 1.0
        match['match_reason'] = 'Exact match'
        all_matches.append(match)

    # Add fuzzy matches
    for match in match_results['fuzzy_matches']:
        match['match_type'] = 'Fuzzy match'
        match['confidence'] = match.get('similarity', 0.8)
        all_matches.append(match)

    # Add unmatched
    for kommune in match_results['no_matches']:
        all_matches.append({
            'main_kommune': kommune,
            'matched_kommune': '',
            'sone': '',
            'county': '',
            'kommunenummer': '',
            'match_type': 'Unmatched',
            'confidence': 0.0,
            'match_reason': 'No match'
        })

    # Create resulting DataFrame and sort
    results_df = pd.DataFrame(all_matches)
    results_df = results_df.sort_values(['match_type', 'confidence'], ascending=[True, False])

    return results_df, match_results


In [42]:
# ================== Part 4: Add Subsidy Zones to Main Data ==================

def get_kommune_zone_mapping(subsidy_zones):
    """Create Kommune to Zone mapping dictionaries"""

    # Create multiple lookup dictionaries to improve match success rate
    zone_dict = {}
    zone_main_dict = {}
    zone_normalized_dict = {}

    for _, row in subsidy_zones.iterrows():
        kommune = row['Kommune']
        zone = row['Sone']

        kommune_lower = clean_kommune_name(kommune)
        kommune_main = extract_main_name(kommune).lower()
        kommune_normalized = normalize_kommune_name(kommune)

        zone_dict[kommune_lower] = zone
        if kommune_main:
            zone_main_dict[kommune_main] = zone
        if kommune_normalized:
            zone_normalized_dict[kommune_normalized] = zone

    return zone_dict, zone_main_dict, zone_normalized_dict

def match_kommune_to_zone(kommune, zone_dict, zone_main_dict, zone_normalized_dict, subsidy_zones):
    """Match a single Kommune to its corresponding Zone"""

    kommune_clean = clean_kommune_name(kommune)
    kommune_main = extract_main_name(kommune).lower()
    kommune_normalized = normalize_kommune_name(kommune)

    if kommune_clean == "":
        return None, "Empty Value"

    # 1. Exact match
    if kommune_clean in zone_dict:
        return zone_dict[kommune_clean], "Exact Match"

    # 2. Handle known special cases
    special_matches = {
        'deatnu tana': 'Deatnu-Tana',
        'unjargga nesseby': 'Unjárga-Nesseby',
        'porsanger porsángu porsanki': 'Porsanger-Porsáŋgu-Porsanki',
        'guovdageaidnu kautokeino': 'Guovdageaidnu-Kautokeino',
        'karasjohka karasjok': 'Kárášjohka-Karasjok',
        'gáivuotna kåfjord': 'Gáivuotna-Kåfjord-Kaivuono',
        'fauske': 'Fauske-Fuossko',
        'harstad': 'Harstad - Hárstták',
        'lavangen': 'Loabák - Lavangen',
        'røyrvik': 'Raarvihke - Røyrvik',
        'snåsa': 'Snåase-Snåsa',
        'storfjord': 'Storfjord-Omasvuotna-Omasvuono',
    }

    if kommune_clean in special_matches:
        target_kommune = special_matches[kommune_clean]
        for _, row in subsidy_zones.iterrows():
            if row['Kommune'] == target_kommune:
                return row['Sone'], "Special Rule Match"

    # 3. Handle Kommunes containing "unntatt tidligere" or "tidligere"
    if kommune_clean in ['heim', 'inderøy', 'indre fosen', 'steinkjer']:
        for _, row in subsidy_zones.iterrows():
            if row['Kommune'].lower().startswith(kommune_clean):
                return row['Sone'], "Partial Kommune Match"

    # 4. Handle special cases for Våler and Herøy
    if kommune_main in ['våler', 'herøy']:
        if '(innlandet)' in kommune_clean or 'våler (innlandet)' == kommune:
            target = 'Våler (Hedm.)Våler (Hedm.)'
        elif '(viken)' in kommune_clean or 'våler (viken)' == kommune:
            target = 'Våler (Østf.)'
        elif '(møre og romsdal)' in kommune_clean or 'herøy (møre og romsdal)' == kommune:
            target = 'Herøy (M. og R.)'
        elif '(nordland)' in kommune_clean or 'herøy (nordland)' == kommune:
            target = 'Herøy (Nordl.)'
        else:
            target = None

        if target:
            for _, row in subsidy_zones.iterrows():
                if row['Kommune'] == target:
                    return row['Sone'], "County-Specific Match"

    # 5. Main name match
    if kommune_main and kommune_main in zone_main_dict:
        return zone_main_dict[kommune_main], "Main Name Match"

    # 6. Normalized name match
    if kommune_normalized in zone_normalized_dict:
        return zone_normalized_dict[kommune_normalized], "Normalized Match"

    # 7. Fuzzy match (high similarity)
    best_match = None
    best_similarity = 0

    for _, row in subsidy_zones.iterrows():
        subsidy_kommune = row['Kommune']

        similarities = [
            calculate_similarity(kommune, subsidy_kommune),
            calculate_similarity(kommune_main, extract_main_name(subsidy_kommune)),
            calculate_similarity(kommune_normalized, normalize_kommune_name(subsidy_kommune))
        ]

        max_similarity = max(similarities)

        if max_similarity > best_similarity and max_similarity > 0.75:
            best_similarity = max_similarity
            best_match = row

    if best_match is not None:
        return best_match['Sone'], f"High-Similarity Match ({best_similarity:.2f})"

    return None, "Unmatched"


In [43]:
def add_subsidy_zones_to_main_data(main_data, subsidy_zones):
    """
    Main function: Add subsidy zones to main data
    """

    print("\n[Step 3] Adding subsidy zones to main data...")
    print(f"Main_Data_Bank has {len(main_data)} rows")
    print(f"norway_subsidy_zones has {len(subsidy_zones)} rows")

    # Create mapping dictionaries
    zone_dict, zone_main_dict, zone_normalized_dict = get_kommune_zone_mapping(subsidy_zones)

    # Add new columns to main_data
    print("\nMatching Kommunes to Subsidy Zones...")
    zones = []
    match_types = []

    # Statistics
    match_stats = {
        "Exact Match": 0,
        "Special Rule Match": 0,
        "Partial Kommune Match": 0,
        "County-Specific Match": 0,
        "Main Name Match": 0,
        "Normalized Match": 0,
        "High-Similarity Match": 0,
        "Unmatched": 0,
        "Empty Value": 0
    }

    # Process each row
    for idx, row in main_data.iterrows():
        kommune = row['Kommune']
        zone, match_type = match_kommune_to_zone(kommune, zone_dict, zone_main_dict,
                                                  zone_normalized_dict, subsidy_zones)
        zones.append(zone)
        match_types.append(match_type)

        # Update stats
        if match_type.startswith("High-Similarity Match"):
            match_stats["High-Similarity Match"] += 1
        else:
            match_stats[match_type] += 1

        # Progress
        if (idx + 1) % 5000 == 0:
            print(f"Processed {idx + 1}/{len(main_data)} rows...")

    # Add columns
    main_data['Subsidy_Zone'] = zones
    main_data['Match_Type'] = match_types

    # Print stats
    print("\nMatching Statistics:")
    print("-" * 40)
    total_rows = len(main_data)
    for match_type, count in match_stats.items():
        if count > 0:
            percentage = count / total_rows * 100
            print(f"{match_type}: {count} ({percentage:.1f}%)")

    # Calculate match rate
    matched_count = total_rows - match_stats["Unmatched"] - match_stats["Empty Value"]
    match_rate = matched_count / (total_rows - match_stats["Empty Value"]) * 100
    print(f"\nTotal Match Rate: {match_rate:.1f}% (excluding empty values)")

    # Print summary of fuzzy matches for review
    print("\nHigh-Similarity Matches requiring review:")
    fuzzy_df = main_data[main_data['Match_Type'].str.contains('High-Similarity Match')]
    if len(fuzzy_df) > 0:
        fuzzy_summary = fuzzy_df.groupby(['Kommune', 'Subsidy_Zone', 'Match_Type']).size().reset_index(name='Count')
        for _, row in fuzzy_summary.head(10).iterrows():
            print(f"{row['Kommune']} → Zone {row['Subsidy_Zone']} {row['Match_Type']} (Total {row['Count']} rows)")
        if len(fuzzy_summary) > 10:
            print(f"... and {len(fuzzy_summary) - 10} more unique matches")

    # Print unmatched Kommunes
    unmatched_df = main_data[main_data['Match_Type'] == 'Unmatched']
    if len(unmatched_df) > 0:
        print("\nUnmatched Kommunes:")
        unmatched_kommuner = unmatched_df['Kommune'].unique()
        for idx, kommune in enumerate(unmatched_kommuner[:10], 1):
            print(f"{idx}. {kommune}")
        if len(unmatched_kommuner) > 10:
            print(f"... and {len(unmatched_kommuner) - 10} more unmatched Kommunes")

    # Drop the Match_Type column before returning (only keep Subsidy_Zone)
    main_data = main_data.drop(columns=['Match_Type'])

    return main_data


In [44]:
# ================== Main Program ==================

def main():
    """Main program execution"""

    # Step 1: Read main data
    print("\n[Step 1] Reading main data file...")
    main_data_path = "/content/drive/MyDrive/BT4222/data/Main_Data_Bank.xlsx"
    from google.colab import drive
    drive.mount('/content/drive')
    #main_data = pd.read_excel("/content/drive/MyDrive/BT4222/data/Main_Data_Bank.xlsx")
    main_data = pd.read_excel(main_data_path)
    print(f"✓ Successfully loaded {len(main_data)} rows from main data")

    # Check if Subsidy_Zone column already exists
    if 'Subsidy_Zone' in main_data.columns:
        print("\n[INFO] Subsidy_Zone column already exists in the data")
        # Check how many are filled
        filled = main_data['Subsidy_Zone'].notna().sum()
        print(f"Currently {filled}/{len(main_data)} rows have subsidy zones filled")

        if filled == len(main_data):
            print("\nAll rows already have subsidy zones. No action needed.")
            return
        else:
            print("\nSome rows missing subsidy zones. Will update missing values...")
            # Remove the existing column to recalculate all
            main_data = main_data.drop(columns=['Subsidy_Zone'])

    # Step 2: Scrape subsidy zone data
    subsidy_zones = scrape_subsidy_zones()

    # Step 3: Run Kommune matching analysis (optional - for information)
    print("\n" + "=" * 60)
    print("Kommune Matching Analysis")
    print("=" * 60)
    results_df, match_results = match_kommune_data(main_data, subsidy_zones)

    # Step 4: Add subsidy zones to main data
    print("\n" + "=" * 60)
    print("Adding Subsidy Zones to Main Data")
    print("=" * 60)
    main_data_with_zones = add_subsidy_zones_to_main_data(main_data, subsidy_zones)

    # Step 5: Save updated data
    print("\n[Step 4] Saving updated data...")
    main_data_with_zones.to_excel(main_data_path, index=False)
    print(f"✓ Successfully saved updated data to {main_data_path}")

    print("\n" + "=" * 80)
    print("Processing completed successfully!")
    print(f"Updated {main_data_path} with subsidy zone information")
    print("=" * 80)


In [45]:
# Run the main program
if __name__ == "__main__":
    main()



[Step 1] Reading main data file...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Successfully loaded 37053 rows from main data

[Step 1] Scraping subsidy zone data...
✓ Successfully scraped 364 rows of subsidy zone data

Kommune Matching Analysis

[Step 2] Starting Kommune matching analysis...
Main_Data_Bank contains 37053 rows of data
norway_subsidy_zones contains 364 rows of data

Unique Kommunes in Main_Data_Bank: 351
Unique Kommunes in norway_subsidy_zones: 364

Starting Kommune matching...

Match results summary:
Exact matches: 328
Fuzzy matches: 23
No matches: 0

Adding Subsidy Zones to Main Data

[Step 3] Adding subsidy zones to main data...
Main_Data_Bank has 37053 rows
norway_subsidy_zones has 364 rows

Matching Kommunes to Subsidy Zones...
Processed 5000/37053 rows...
Processed 10000/37053 rows...
Processed 15000/37053 rows...
Processed 20000/37053 rows...
Processed 25000/37053 rows...
Proce